![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5D: Copilot Vision and Task Execution

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional real vision-model section if image-capable API access is available.</td></tr>
<tr><td align="left">Main output</td><td>Simulate a vision-assisted copilot that converts observations into safe tasks.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05d-overview)
2. [Setup and Background](#m05d-background)
3. [Core Concepts](#m05d-data)
4. [Guided Implementation](#m05d-workflow)
5. [Testing and Analysis](#m05d-testing)
6. [Student Tasks](#m05d-tasks)
7. [Submission and Reflection](#m05d-submission)

---

<a id="m05d-overview"></a>

### 1. Overview and Learning Goals

This session is **M05D: Copilot Vision and Task Execution**. So far in M05 your systems have answered questions. This session looks at systems that *act*: copilots that observe a screen or an image, propose a task, and — only after a human confirms — execute it. The stakes change immediately. A wrong answer can be ignored; a wrong click, deletion or submission cannot be un-happened. Everything in this notebook is shaped by that difference.

The central theme is:

```text
Simulate a vision-assisted copilot that converts observations into safe tasks.
```

A useful analogy is a driving instructor sitting in the passenger seat. The instructor sees the same road you do (vision observation), says "there is a stop sign ahead, you should brake" (task plan), and waits for you to act (human confirmation). A good instructor never grabs the wheel uninvited — and a copilot that executes tasks without confirmation is a system that has grabbed the wheel.

The full loop looks like this:

```text
        +-------------------------------------------------------+
        |                                                       |
        v                                                       |
[ Observe screen or image ]                                     |
        |                                                       |
        v                                                       |
[ Propose task plan ] --(outside safety boundary)--> refuse     |
        |                                            + explain  |
        v                                                       |
[ Human confirms? ] ----no----> stop, or revise the plan        |
        |                                                       |
       yes                                                      |
        |                                                       |
        v                                                       |
[ Execute safe, approved action ] ------------------------------+
                (then observe the result and continue the loop)
```

The mandatory workflow in this notebook simulates the observe-and-propose part of this loop with plain Python and text descriptions standing in for real images. No real vision model is called and no real action is executed — which is exactly the point: the safety structure (validation, evidence, refusal, limitation statements) must exist *before* a real model or real execution is attached, not after.

The concepts used in this session are:

```text
1. vision observation
2. task plan
3. human confirmation
4. safe execution
5. limitation statement
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal and failure cases, and explain how the design would change if a real model or external package were added.


<a id="m05d-background"></a>

### 2. Setup and Background

#### 2.1 Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow — and for a task-executing copilot, that discipline is what stands between "assistant" and "accident".

A weak copilot design does this:

```text
Screenshot --> One large prompt --> Model output --> Action executed
                                                     (no checkpoint anywhere)
```

This is simple, but it hides too many decisions. Was the observation understood correctly? Was the proposed action within the allowed boundary? Did a human ever agree to it? By the time you can ask those questions, the action has already happened.

A stronger design separates observation, planning and execution, with checkpoints between them:

```text
Observation (described)
      |
      v
[ Validate request ]       is this observation request well-formed and allowed?
      |
      v
[ Match approved knowledge ]  what do we know about handling this situation?
      |
      v
[ Propose task plan ]      a structured plan with evidence and limitations
      |
      v
[ Human confirmation ]     the plan is REVIEWED before anything executes
      |
      v
[ Safe execution ]         only approved, reversible-where-possible actions
```

For **Copilot Vision and Task Execution**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>What it means here</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">vision observation</td><td>What the copilot can see — here simulated by text descriptions, because the safety structure must not depend on how good the eyes are.</td></tr>
<tr><td align="left">task plan</td><td>A structured proposal built only from approved knowledge, with evidence attached — never a raw "do this" instruction.</td></tr>
<tr><td align="left">human confirmation</td><td>The checkpoint between plan and action. In this notebook every result stops at the plan stage, which is the confirmation boundary.</td></tr>
<tr><td align="left">safe execution</td><td>Only approved actions, only after confirmation; unsafe requests are refused at validation, before planning even starts.</td></tr>
<tr><td align="left">limitation statement</td><td>Every plan says what it is based on and what it does not know — vision systems misread screens, and a plan should admit that.</td></tr>
</tbody>
</table>

</div>

The mandatory workflow uses a local simulation because local simulations make the control structure visible. A real vision model can be added later, but it should not replace validation, inspection, tests, limitations, and human confirmation — it only replaces the eyes.


<a id="m05d-setup"></a>

#### 2.2 Environment and Safety

The mandatory part uses standard Python only, so it runs identically in Colab and local Jupyter, with no packages to install and no API key to manage. The optional section later mentions image-capable model calls, but those are not required.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

Point 3 is the defining rule for this session: the workflow you build proposes tasks but never executes anything. Real execution belongs behind a human-confirmation checkpoint that this lab deliberately does not cross. If you find yourself wanting to add code that clicks, deletes or sends — stop at printing the plan instead.

Run the setup cell below; you should see `Setup complete.` and nothing else.


In [ ]:
# Standard library only: the mandatory copilot simulation needs no
# installation, no credentials and -- most importantly -- has no ability
# to execute real actions. It can only read data and print plans.

import json  # pretty-printing items and structured plans
import re    # tokenising text for lexical matching
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")


<a id="m05d-data"></a>

### 3. Core Concepts

#### 3.1 Approved Local Data

The local data below is synthetic teaching data for this practical. It is not private data. It is deliberately small so that you can inspect every item and understand why the workflow produced a result.

Think of each item as one entry in the copilot's approved playbook: knowledge about how observations should be turned into tasks, and where the boundaries lie. Each item has:

```text
item_id:    stable identifier, cited as the evidence for a plan
title:      short title shown with selection results
content:    the approved teaching content plans are built from
tags:       labels that give the lexical matcher extra vocabulary
risk_level: low / medium / high -- how carefully a plan built on this
            item should be reviewed before confirmation
```

The `risk_level` field previews a real design pattern: production copilots grade proposed actions, and higher-risk actions demand stronger confirmation (or are excluded entirely). In this lab the field is informational, but Task 2 invites you to build on it.

In a production system, equivalent data might come from public documentation, approved playbooks, or authorised internal systems. This practical does not use those live sources. Run the next cell and confirm it reports three items and prints the first one in full.


In [ ]:
# Three small approved playbook entries for the copilot simulation.
# Small data is a feature: when the workflow builds a plan, you can read
# the entire evidence base and judge whether the plan is supported.

LOCAL_ITEMS = [
    {
        "item_id": "M05D-001",
        "title": "Vision Observation Basics",
        "content": "A vision observation should describe visible evidence without inferring hidden state or claiming access beyond the image.",
        "tags": ["vision", "observation", "visible", "evidence", "uncertainty"],
        "risk_level": "low"
    },
    {
        "item_id": "M05D-002",
        "title": "Task Plan Practice",
        "content": "A safe task plan names the proposed action, supporting observation and risk, then stops before execution.",
        "tags": ["task_plan", "proposed_action", "evidence", "risk", "stop"],
        "risk_level": "low"
    },
    {
        "item_id": "M05D-003",
        "title": "Human Confirmation Safety",
        "content": "Human confirmation is required before any consequential action; unclear or unsafe requests remain unexecuted.",
        "tags": ["human_confirmation", "approval", "safety", "execution", "refusal"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as an approved playbook in a real copilot. The purpose is not to cover every observation a copilot might meet. The purpose is to make the workflow observable: you can predict which entry a request should match and what kind of plan should come back, run it, and check. A copilot whose decisions you cannot predict is a copilot you cannot safely confirm.


<a id="m05d-workflow"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Workflow

The workflow has four functions, mirroring the observe-and-propose half of the copilot loop:

```text
1. validate_request        -- is this observation request well-formed and allowed?
2. select_relevant_items   -- which approved playbook entries apply?
3. build_structured_result -- what plan can honestly be proposed from them?
4. run_local_workflow      -- the orchestrator that wires 1-3 together
```

Every request ends in exactly one of four outcomes, and the tests later check all four:

```text
completed             relevant playbook entries found; a grounded plan returned
insufficient_context  request allowed, but the playbook does not cover it
refused               request asks for something outside the safety boundary
ok = False            malformed input (empty request, invalid top_k)
```

Notice where the workflow *ends*: at a structured result — a proposed plan with evidence and limitations. Execution is deliberately absent. In the full copilot loop, this output is what a human reads at the confirmation checkpoint. Keeping the boundary in the architecture (rather than in a model's good intentions) is the design lesson of this session.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.


In [ ]:
def normalise_text(text: str) -> str:
    # Collapse whitespace and lowercase, so matching is not affected by
    # formatting differences in observation requests.
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Returning [] for non-string input means later stages never crash on
    # unexpected types; they simply find no matching terms.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # The copilot checks the request BEFORE any planning happens. Two
    # different negative outcomes are kept deliberately distinct:
    #   ok = False      -> malformed input (empty / wrong type): a caller
    #                      bug that should be fixed, not planned around.
    #   allowed = False -> well-formed but unsafe: the copilot refuses to
    #                      plan the task at all, and says why.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()

    # A real copilot would use a maintained action policy, not a keyword
    # list. This list is a teaching stand-in, and every term names an
    # action or target that must never reach the planning stage: refusing
    # here is cheaper and safer than refusing after a plan exists.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }


In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # The copilot may only plan from its approved playbook -- this function
    # finds the entries that apply. Guard top_k first: silently accepting
    # top_k=0 would return empty results that look like "playbook does not
    # cover this" and hide the caller's bug.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        # Title, content and tags are merged into one searchable string,
        # so a request can match an entry through any of the three fields.
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        # Set intersection counts *distinct* shared terms -- the same
        # transparent lexical scoring used in M05A and M05B.
        score = len(request_terms.intersection(item_terms))
        if score > 0:
            # score > 0 filter: an observation the playbook does not cover
            # matches nothing, which triggers the insufficient-context
            # branch downstream -- the copilot admits the gap instead of
            # improvising a plan.
            selected = dict(item)          # copy: never mutate the playbook
            selected["score"] = score      # keep the evidence visible
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    return {"ok": True, "error": None, "result": scored[:top_k]}


In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # The empty-selection branch is the copilot's most important safety
    # feature: when the playbook does not cover an observation, the honest
    # output is "insufficient context" -- never an improvised task plan.
    # An improvised plan that gets confirmed by a trusting human is how
    # copilot systems cause real damage.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "Copilot Vision and Task Execution. The result is based only on selected local evidence."
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            # The selected entries travel with the plan: they are what a
            # human reads at the confirmation checkpoint to judge it.
            "selected_items": selected_items,
            # Limitation statements are attached even to successful plans.
            # Vision systems misread screens; a plan should say what it is
            # based on so the human can catch a misreading.
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }


In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # The orchestrator wires the observe-and-propose steps together and
    # routes each request to exactly one outcome: completed (a plan ready
    # for human confirmation), insufficient_context, refused, or ok=False.
    # Note what is NOT here: no execution step. The workflow ends where
    # the human-confirmation checkpoint begins.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation                     # malformed input: stop immediately

    if not validation["result"]["allowed"]:
        # Refusal is a normal, well-formed outcome (ok stays True): the
        # copilot worked correctly by declining to plan an unsafe task.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # The request is echoed into the result so the plan is a complete,
    # self-describing record -- exactly what a confirmation screen shows.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


example_result = run_local_workflow("How should a copilot turn a screen observation into a safe task plan with human confirmation?", LOCAL_ITEMS)
example_result


<a id="m05d-inspection"></a>

#### 4.2 Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08 — and for a copilot it has a concrete role: inspection here is a rehearsal of the human-confirmation checkpoint. The questions you ask of the printed result are the questions a user must be able to answer before clicking "confirm" on a real proposed action.

For every result, check:

```text
1. Was the request allowed?
2. Which playbook entries were selected, and with what scores?
3. Do the selected entries actually support the proposed result?
4. Did the workflow state its limitations?
5. Did it refuse unsafe requests, with a reason?
```

Question 3 is the one that matters most at a confirmation checkpoint. A plan can have status `completed`, read sensibly, and still rest on a weakly related entry that shared one word with the request. If you cannot trace the plan to its evidence, the correct response at the checkpoint is "no".


In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # Print everything a human would need at the confirmation checkpoint:
    # the outcome, the evidence behind it, and the stated limitations.
    # A copilot that shows only its conclusion cannot be safely confirmed.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)


A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. For a copilot this is not a stylistic preference: the confirmation checkpoint only works if a human can see, in one screen, what is proposed, what it is based on, and what the system does not know.


<a id="m05d-optional"></a>

#### 4.3 Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. If image-capable API access is available, a real vision model could replace the simulated observation — the model would describe a screenshot, and that description would enter the workflow exactly where the text request enters now.

Notice how little that changes:

```text
Screenshot --> [ Real vision model ] --> Observation description
                                                |
                                                v
                              [ Validate ] --> [ Match playbook ] --> [ Propose plan ]
                                                                            |
                                                                            v
                                                            Human confirmation checkpoint
```

The model upgrades the *eyes*; validation, playbook matching, refusal, limitation statements and the confirmation checkpoint all remain. A design in which a better vision model removes the confirmation step is not an upgrade — it is the weak workflow from Section 2 wearing better glasses.

Do not hard-code API keys — if you add a real call, read the key from the environment using the `getpass`/`os.environ` pattern from M05A. Do not send private screenshots to any API. If the optional section is not available, write:

```text
Skipped: optional package/API access not available.
```


In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe: it makes no external calls,
# needs no key, and sends no images anywhere. If you later add a real
# vision-model call, keep this guard structure -- check availability
# first, and degrade to a clear skipped message instead of crashing.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")


<a id="m05d-testing"></a>

### 5. Testing and Analysis

A system that proposes actions must be tested hardest on the cases where it should *not* act. The tests below therefore give the negative behaviours equal weight with the happy path:

```text
1. Normal case:              a covered observation produces a grounded plan
                             with selected evidence.
2. Missing-information case: an uncovered observation produces an explicit
                             insufficient-context result, never an
                             improvised plan.
3. Safety/refusal case:      an unsafe request is refused with a reason
                             before any planning happens.
4. Failure case:             malformed input (empty request, top_k = 0)
                             is rejected with ok = False.
```

If any assertion fails, Python raises `AssertionError` at the failing line — run the same request through `display_workflow_result`, read the `status` field to see which outcome it actually reached, and work out whether the workflow or the expectation is wrong.


In [ ]:
# Normal case: a request the playbook covers should complete with at
# least one selected entry as evidence for the plan.
normal = run_local_workflow("How should a copilot turn a screen observation into a safe task plan with human confirmation?", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Missing-information case: the playbook says nothing about exam rooms,
# so the only honest outcome is insufficient_context with no items --
# a copilot must not improvise a plan for a situation it does not know.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Safety/refusal case: an unsafe request must be refused with a reason,
# before planning. Note ok is still True: refusing correctly is the
# copilot succeeding, not failing.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Failure case: empty request is malformed input, reported with ok=False
# so a calling system can tell "bad call" apart from "safe refusal".
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Failure case: top_k=0 is a caller bug and must be rejected, not treated
# as "the playbook does not cover this".
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")


In [ ]:
# Display the full trace for three contrasting requests: one the playbook
# covers, one it does not, and one that must be refused outright.
for request in [
    "How should a copilot turn a screen observation into a safe task plan with human confirmation?",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))


<a id="m05d-tasks"></a>

### 6. Student Tasks

Complete the tasks below in order — each task builds on the previous one. The mandatory local workflow must run without external API calls and must never execute real actions. Keep your work in clearly labelled cells so a marker can find each piece of evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from Setup through Testing and Analysis without modification.</td><td>Confirms your environment reproduces the reference behaviour — including the refusal and insufficient-context outcomes — before you change anything.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add new observation/task rule</td><td>Append one new approved synthetic playbook entry to <code>LOCAL_ITEMS</code> that maps an observation to a safe task — for example: "if the screen shows an error dialog, the safe task is to read and report the message, not to click through it". Set a sensible <code>risk_level</code>. No private data, no real side effects.</td><td>Writing an observation-to-task rule forces you to decide what a copilot should propose — and, just as importantly, what it should not.</td><td>A code cell showing the complete new entry.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a request whose words overlap your new entry, and show the output with <code>display_workflow_result</code>.</td><td>Verifies your rule is actually matchable and produces the plan you intended, not just stored.</td><td>Displayed result with your <code>item_id</code> among the selected items and status <code>completed</code>.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Write at least three <code>assert</code>-based tests: a normal case (your entry selected, status <code>completed</code>), a missing-information case (uncovered observation gives <code>insufficient_context</code> with no items), and a refusal or failure case (unsafe request gives <code>refused</code>, or empty request gives <code>ok=False</code>).</td><td>A system that proposes actions must be tested hardest on the cases where it should not act.</td><td>A test cell that runs with all assertions passing.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>For one completed result, identify which playbook entry supports the plan and whether anything in the output goes beyond the selected evidence. Note the <code>risk_level</code> of the evidence used.</td><td>This analysis is a rehearsal of the human-confirmation decision: could you responsibly confirm this plan from what is shown?</td><td>A short grounding paragraph in a markdown cell.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>If image-capable API access is available, extend the optional section safely (environment-variable key pattern; no private screenshots). If not, write <code>Skipped: optional package/API access not available</code>.</td><td>Shows that a real vision model replaces only the eyes — the confirmation boundary and refusal rules stay.</td><td>Output, or the skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Explain what this workflow teaches about agentic AI design, focusing on why the workflow stops at a proposed plan instead of executing it.</td><td>Being able to justify the confirmation boundary matters more than reproducing the code.</td><td>150–250 words in a markdown cell.</td></tr>
</tbody>
</table>

</div>


In [ ]:
# Student task starter (Tasks 2 and 3).
#
# Step 1: design one approved synthetic playbook entry that maps an
#         observation to a SAFE task (observe, read, report, summarise --
#         not click, delete, send). Choose tags a request would contain,
#         and set a risk_level you can justify.
# Step 2: append it to LOCAL_ITEMS.
# Step 3: run a request that shares words with your entry, and confirm
#         with display_workflow_result that YOUR item_id is selected and
#         the status is "completed".
#
# Uncomment and adapt the example below.

# new_item = {
#     "item_id": "M05D-004",
#     "title": "Error Dialog Observation Rule",
#     "content": "If the screen shows an error dialog, the safe task is to read and report the exact message, not to click through it or retry blindly.",
#     "tags": ["error_dialog", "observation", "report", "safety"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("What should happen when an error dialog appears?", LOCAL_ITEMS)
# display_workflow_result(result)


<a id="m05d-submission"></a>

### 7. Submission and Reflection

**Required submission items**

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your new observation/task rule.
3. Workflow output showing your rule in effect.
4. At least three added tests using assert statements.
5. Short grounding analysis (including the risk_level note).
6. Optional package/API result or skipped note.
7. 150-250 word reflection.
```

**Quality checks**

Before submitting, restart the runtime, run all cells top to bottom, and confirm:

- Every cell runs without errors in a fresh runtime.
- No API key, password, private data or real screenshot appears anywhere in the notebook.
- Your new entry proposes only safe, observe-and-report style tasks and has a justified <code>risk_level</code>.
- Your three added tests pass, and cover normal, missing-information and refusal/failure cases.
- No cell anywhere in the notebook executes a real external action.

**Debugging guide**

- `AssertionError` in the baseline tests: a cell above was changed or skipped. Restart the runtime and run all cells in order before investigating further.
- Your entry is never selected: print `tokenise(your_request)` and `tokenise(your_entry_content)` and look for shared terms. Lexical matching needs word overlap — adjust the request wording or the entry tags.
- Status is `completed` but the selected entry looks unrelated: the request shares an accidental word with the wrong entry. This is exactly the failure the confirmation checkpoint exists to catch — make the request more specific and mention the episode in your grounding analysis.
- A safe request is being refused: one of the `unsafe_terms` appears inside your wording (for example "send email" inside a longer phrase). Reword the request or refine the rule, and note the trade-off.
- Optional section errors: package or API access is missing. That is expected — write the skipped note and move on.

**Reflection questions**

1. What are the main stages of the copilot workflow, and where does it deliberately stop?
2. Why does the workflow validate the request before any planning happens?
3. What should happen when the playbook does not cover an observation?
4. Why must unsafe requests be refused before planning, rather than filtered after?
5. If a real vision model replaced the simulated observation, what would change — and what must not?

#### Further Readings

- LangChain multimodality concepts: <https://python.langchain.com/docs/concepts/multimodality/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- LangGraph human-in-the-loop concepts: <https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/>
